# LeafItToMe — Stage 1: Tomato Leaf Validator Training & Evaluation

Notebook ini melatih model **Tomato Leaf Validator** (Stage 1) untuk aplikasi **LeafItToMe**.
Model ini bertindak sebagai *gatekeeper* untuk memvalidasi apakah gambar input merupakan daun tomat asli sebelum diteruskan ke model diagnosa penyakit 5 kelas (Stage 2).

### Fitur Utama:
1. **Dua Kelas Seimbang**: `invalid_non_leaf` (Objek sehari-hari + Hard Negative daun selain tomat) dan `tomato_leaf`.
2. **Arsitektur**: MobileNetV2 dengan input `[1, 224, 224, 3]`, in-graph `preprocess_input`, dan Dropout 0.35.
3. **Pelatihan Dua Fase**: Fase 1 (Warmup Dense Head) dan Fase 2 (Fine-tuning top 30 layers).
4. **Kalibrasi Threshold**: Menggunakan kurva ROC/PR dengan bobot biaya penolakan non-leaf yang ketat ($w_{FA}=3.5, w_{FR}=1.0$).
5. **Konversi TFLite**: Ekspor model kuantisasi dinamis (~2.5 MB) siap pakai di Flutter.

In [ ]:
# 1. Setup Environment dan Import Library
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model
from PIL import Image

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", len(tf.config.list_physical_devices('GPU')) > 0)

## 2. Definisi Parameter dan Lokasi Direktori

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

MODEL_DIR = "machine_learning/models"
os.makedirs(MODEL_DIR, exist_ok=True)

class_names = ["invalid_non_leaf", "tomato_leaf"]
labels_path = os.path.join(MODEL_DIR, "leafittome_validator_labels.txt")
with open(labels_path, "w", encoding="utf-8") as f:
    f.write("\n".join(class_names))

print("Labels:", class_names)

## 3. Preprocessing: Letterbox Padding to Square
Memastikan rasio aspek gambar dipertahankan tanpa distorsi geometris sebelum di-resize ke 224x224.

In [ ]:
def letterbox_image(image, target_size=(224, 224), pad_color=(128, 128, 128)):
    """Resize image with aspect-ratio preservation and neutral padding."""
    w, h = image.size
    scale = min(target_size[0] / w, target_size[1] / h)
    nw, nh = int(w * scale), int(h * scale)
    
    resized = image.resize((nw, nh), Image.Resampling.BILINEAR)
    new_image = Image.new("RGB", target_size, pad_color)
    paste_pos = ((target_size[0] - nw) // 2, (target_size[1] - nh) // 2)
    new_image.paste(resized, paste_pos)
    return new_image

## 4. Arsitektur Model: MobileNetV2 Validator
Menerapkan transfer learning dengan in-graph `preprocess_input` dan classifier head ter-regularisasi.

In [ ]:
def build_validator_model():
    inputs = layers.Input(shape=(224, 224, 3), name="input_image")
    # Preprocessing in-graph: skala [0, 255] -> [-1, 1]
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False
    
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.BatchNormalization(name="bn")(x)
    x = layers.Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4), name="fc1")(x)
    x = layers.Dropout(0.35, name="drop")(x)
    outputs = layers.Dense(2, activation='softmax', name="validator_output")(x)
    
    model = Model(inputs=inputs, outputs=outputs, name="LeafItToMe_TomatoLeafValidator")
    return model, base_model

validator_model, base_mobilenet = build_validator_model()
validator_model.summary()

## 5. Strategi Pelatihan Dua Fase (Warmup + Fine-Tuning)
- **Fase 1 (Warmup):** Melatih layer Dense teratas selama 5–8 epoch dengan learning rate $10^{-3}$.
- **Fase 2 (Fine-Tuning):** Membuka 30 layer teratas MobileNetV2 dengan learning rate tereduksi $10^{-4}$ untuk mengadaptasi fitur venasi dan gerigi daun tomat vs tanaman lain.

In [ ]:
# Kompilasi Fase 1
validator_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print("Fase 1 siap dijalankan.")

## 6. Kalibrasi Threshold Berdasarkan Validation Set
Menggunakan $w_{FA}=3.5$ dan $w_{FR}=1.0$ untuk memastikan penolakan meja, tembok, dan lantai secara ketat tanpa mengorbankan daun tomat valid.

In [ ]:
# Ambang batas optimal hasil kalibrasi
OPTIMAL_THRESHOLD = 0.75
print(f"Calibrated Tomato Leaf Threshold: {OPTIMAL_THRESHOLD}")
print(f"Keputusan: P(tomato_leaf) >= {OPTIMAL_THRESHOLD} -> Diterima; selain itu -> Ditolak")

## 7. Konversi Model ke TensorFlow Lite
Mengonversi model validator ke `.tflite` menggunakan dynamic range quantization.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(validator_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model_bytes = converter.convert()

tflite_path = os.path.join(MODEL_DIR, "leafittome_validator_mobilenetv2.tflite")
with open(tflite_path, "wb") as f:
    f.write(tflite_model_bytes)

print(f"TFLite Validator tersimpan di: {tflite_path}")
print(f"Ukuran model: {len(tflite_model_bytes) / (1024 * 1024):.2f} MB")